In [1]:
import pandas as pd

# Load the dataset
df = pd.read_csv("global_unemployment.csv")

# Display variable types
print(df.dtypes)
print(f"\nShape: {df.shape}")

country_name       object
indicator_name     object
sex                object
age_group          object
age_categories     object
2014              float64
2015              float64
2016              float64
2017              float64
2018              float64
2019              float64
2020              float64
2021              float64
2022              float64
2023              float64
2024              float64
dtype: object

Shape: (1134, 16)


In [3]:
# Define year columns (numeric unemployment rates)
year_cols = [str(y) for y in range(2014, 2025)]

# Calculate IQR per year column
for col in year_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f"{col} → outliers: {len(outliers)} | bounds: [{lower:.2f}, {upper:.2f}]")

2014 → outliers: 74 | bounds: [-12.74, 31.73]
2015 → outliers: 75 | bounds: [-12.17, 30.93]
2016 → outliers: 81 | bounds: [-11.84, 30.25]
2017 → outliers: 74 | bounds: [-11.84, 29.73]
2018 → outliers: 86 | bounds: [-10.83, 27.85]
2019 → outliers: 83 | bounds: [-11.08, 27.91]
2020 → outliers: 79 | bounds: [-12.14, 31.79]
2021 → outliers: 76 | bounds: [-11.94, 30.97]
2022 → outliers: 79 | bounds: [-11.23, 28.19]
2023 → outliers: 81 | bounds: [-10.68, 27.07]
2024 → outliers: 84 | bounds: [-10.38, 26.53]


In [4]:
# ── STEP 1 : Confirm missing values location ──────────────────────────────────
year_cols = [str(y) for y in range(2014, 2025)]

missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0])
print(f"\nTotal missing values: {df.isnull().sum().sum()}")

# ── STEP 2 : Identify affected countries and segments ────────────────────────
missing_rows = df[df[year_cols].isnull().any(axis=1)]
print("\nAffected rows:")
print(missing_rows[['country_name','sex','age_categories',
                     '2021','2022','2023','2024']].to_string())

Missing values per column:
2022     6
2023    12
2024    12
dtype: int64

Total missing values: 30

Affected rows:
                 country_name     sex age_categories    2021    2022  2023  2024
756   Palestinian Territories  Female          Youth  64.249  56.709   NaN   NaN
757   Palestinian Territories  Female         Adults  37.921  36.385   NaN   NaN
758   Palestinian Territories  Female       Children  42.551  40.045   NaN   NaN
759   Palestinian Territories    Male          Youth  36.866  31.563   NaN   NaN
760   Palestinian Territories    Male         Adults  17.955  16.772   NaN   NaN
761   Palestinian Territories    Male       Children  22.224  20.186   NaN   NaN
1056                  Ukraine  Female          Youth  20.412     NaN   NaN   NaN
1057                  Ukraine  Female         Adults   9.519     NaN   NaN   NaN
1058                  Ukraine  Female       Children  10.143     NaN   NaN   NaN
1059                  Ukraine    Male          Youth  18.085     NaN   NaN 

In [5]:
# ── STEP 3 : Imputation by forward fill ──────────────────────────────────────
# Justification : Palestinian Territories and Ukraine are conflict zones
# Data collection was interrupted — not a structural data error
# Forward fill propagates the last known value (2022 for Palestine, 2021 for Ukraine)
# This preserves trend continuity without introducing artificial values
# Method chosen : ffill() — respects temporal order, no distortion of the series

df[year_cols] = df[year_cols].ffill(axis=1)

# ── STEP 4 : Verify correction ───────────────────────────────────────────────
print(f"Missing values after correction: {df.isnull().sum().sum()}")

# Spot check : Palestinian Territories and Ukraine
check = df[df['country_name'].isin(['Palestinian Territories', 'Ukraine'])]
print(check[['country_name','sex','age_categories',
             '2021','2022','2023','2024']].to_string())

Missing values after correction: 0
                 country_name     sex age_categories    2021    2022    2023    2024
756   Palestinian Territories  Female          Youth  64.249  56.709  56.709  56.709
757   Palestinian Territories  Female         Adults  37.921  36.385  36.385  36.385
758   Palestinian Territories  Female       Children  42.551  40.045  40.045  40.045
759   Palestinian Territories    Male          Youth  36.866  31.563  31.563  31.563
760   Palestinian Territories    Male         Adults  17.955  16.772  16.772  16.772
761   Palestinian Territories    Male       Children  22.224  20.186  20.186  20.186
1056                  Ukraine  Female          Youth  20.412  20.412  20.412  20.412
1057                  Ukraine  Female         Adults   9.519   9.519   9.519   9.519
1058                  Ukraine  Female       Children  10.143  10.143  10.143  10.143
1059                  Ukraine    Male          Youth  18.085  18.085  18.085  18.085
1060                  Ukraine 

In [6]:
# ── STEP 5 : Document imputation for auditability ────────────────────────────
# Palestinian Territories : 2023 and 2024 filled from 2022 (last available year)
# Ukraine                 : 2022, 2023 and 2024 filled from 2021 (last available year)
# Both countries : conflict context → data collection interrupted, not structural error
# Decision : forward fill retained over median/mean imputation
# → mean would dilute extreme conflict-zone values, distorting cross-country comparison

print("Imputation applied : forward fill on conflict-zone missing values")
print("Affected countries : Palestinian Territories (2023–2024), Ukraine (2022–2024)")
print("Audit complete. Dataset is ready for the next step.")

Imputation applied : forward fill on conflict-zone missing values
Affected countries : Palestinian Territories (2023–2024), Ukraine (2022–2024)
Audit complete. Dataset is ready for the next step.


In [7]:
# Rename columns for business readability (Power BI / SQL compatible)
df = df.rename(columns={
    "country_name"  : "country",           # lighter snake_case, already clear
    "sex"           : "gender",            # standard ILO/HR decisional term
    "age_group"     : "age_bracket",       # readable in a Power BI slicer
    "age_categories": "population_segment" # explicit for non-technical audience
})

# Rename year columns → rate_YYYY format to clarify they are annual rates
year_cols = {str(y): f"rate_{y}" for y in range(2014, 2025)}
df = df.rename(columns=year_cols)

# Verify renaming
print(df.columns.tolist())
print(df.head(2))

['country', 'indicator_name', 'gender', 'age_bracket', 'population_segment', 'rate_2014', 'rate_2015', 'rate_2016', 'rate_2017', 'rate_2018', 'rate_2019', 'rate_2020', 'rate_2021', 'rate_2022', 'rate_2023', 'rate_2024']
       country                    indicator_name  gender age_bracket  \
0  Afghanistan  Unemployment rate by sex and age  Female       15-24   
1  Afghanistan  Unemployment rate by sex and age  Female         25+   

  population_segment  rate_2014  rate_2015  rate_2016  rate_2017  rate_2018  \
0              Youth     13.340     15.974     18.570     21.137     20.649   
1             Adults      8.576      9.014      9.463      9.920     11.223   

   rate_2019  rate_2020  rate_2021  rate_2022  rate_2023  rate_2024  
0     20.154     21.228     21.640     30.561     32.200     33.332  
1     12.587     14.079     14.415     23.818     26.192     28.298  


In [8]:
# Define renamed year columns
rate_cols = [f"rate_{y}" for y in range(2014, 2025)]

# --- Average unemployment rate over the full period (2014–2024)
df["rate_avg_2014_2024"] = df[rate_cols].mean(axis=1).round(2)

# --- Most recent unemployment rate (2024)
df["rate_latest"] = df["rate_2024"]

# --- COVID impact: difference between 2020 and 2019 rates
df["covid_impact"] = (df["rate_2020"] - df["rate_2019"]).round(2)

# --- Decadal trend: difference between 2024 and 2014 rates
df["decadal_trend"] = (df["rate_2024"] - df["rate_2014"]).round(2)

# --- Trend direction: label based on decadal trend sign
df["trend_direction"] = df["decadal_trend"].apply(
    lambda x: "Improving" if x < 0 else ("Stable" if x == 0 else "Worsening")
)

# --- Risk level: segmentation based on ILO thresholds applied to rate_latest
def classify_risk(rate):
    if rate < 5:    return "Low"
    elif rate < 15: return "Medium"
    elif rate < 30: return "High"
    else:           return "Critical"

df["risk_level"] = df["rate_latest"].apply(classify_risk)

# --- Gender gap: Female rate minus Male rate in 2024 (per country & age_bracket)
female = df[df["gender"] == "Female"][["country", "age_bracket", "rate_2024"]].copy()
male   = df[df["gender"] == "Male"][["country", "age_bracket", "rate_2024"]].copy()

# Merge female and male on country + age_bracket
gap_df = female.merge(male, on=["country", "age_bracket"], suffixes=("_f", "_m"))

# Compute gap: positive = women more unemployed than men
gap_df["gender_gap"] = (gap_df["rate_2024_f"] - gap_df["rate_2024_m"]).round(2)

# Merge gender gap back into main dataframe
df = df.merge(
    gap_df[["country", "age_bracket", "gender_gap"]],
    on=["country", "age_bracket"],
    how="left"
)

# Verify new features
print(df[["country", "gender", "population_segment",
          "rate_avg_2014_2024", "rate_latest",
          "covid_impact", "decadal_trend",
          "trend_direction", "risk_level",
          "gender_gap"]].head(6))

print(f"\nDataset shape after feature engineering: {df.shape}")

       country  gender population_segment  rate_avg_2014_2024  rate_latest  \
0  Afghanistan  Female              Youth               22.62       33.332   
1  Afghanistan  Female             Adults               15.23       28.298   
2  Afghanistan  Female           Children               18.15       30.956   
3  Afghanistan    Male              Youth               14.96       19.770   
4  Afghanistan    Male             Adults                8.99       13.087   
5  Afghanistan    Male           Children               10.89       15.296   

   covid_impact  decadal_trend trend_direction risk_level  gender_gap  
0          1.07          19.99       Worsening   Critical       13.56  
1          1.49          19.72       Worsening       High       15.21  
2          1.37          20.65       Worsening   Critical       15.66  
3          0.09          10.56       Worsening       High       13.56  
4          0.77           6.62       Worsening     Medium       15.21  
5          0.53      

In [9]:
# Drop column with no analytical value (single constant value across all rows)
df = df.drop(columns=["indicator_name"])

# Verify the column is removed
print(f"Columns after drop: {df.columns.tolist()}")

# Confirm final shape
print(f"\nFinal dataset shape: {df.shape}")

Columns after drop: ['country', 'gender', 'age_bracket', 'population_segment', 'rate_2014', 'rate_2015', 'rate_2016', 'rate_2017', 'rate_2018', 'rate_2019', 'rate_2020', 'rate_2021', 'rate_2022', 'rate_2023', 'rate_2024', 'rate_avg_2014_2024', 'rate_latest', 'covid_impact', 'decadal_trend', 'trend_direction', 'risk_level', 'gender_gap']

Final dataset shape: (1134, 22)


In [10]:
df.to_excel("global_unemployment.xlsx",index=False,sheet_name="global_unemployment")

In [11]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus

user = "root"
password = "mysql2026"
host = "localhost"
port = 3306
db = "esmel"

engine = create_engine(
    f"mysql+pymysql://{user}:{password}@{host}:{port}/{db}",
    echo=False
)

In [12]:
engine.connect()

In [13]:
df.to_sql(
    name="global_unemployment",
    con=engine,
    if_exists="replace",
    index=False
)

1134